In [1]:
# You can copy the folder structure of the previous analysis in this way:

# rsync -av -f"+ */" -f"- *" "/nfs/lab/projects/islet_multiomics_stress/analysis/CellChat/Results/May2022/" "/nfs/lab/projects/islet_multiomics_stress/analysis/CellChat/Results/May2022.SCT"

# Installation
It's suggested to do directly on terminal using the same R version of the notebook

In [2]:
#devtools::install_github("sqjin/C§ellChat")
#install.packages('NMF')
#devtools::install_github("jokergoo/circlize")
#devtools::install_github("jokergoo/ComplexHeatmap")

## Load reticulate library and import anndata module
The anndata module in python is used to load .h5ad files in R </br>
Install the anndata module in python withing the correct conda enviroment before launching.

In [3]:
#essential reticulate functions (must run first)
Sys.setenv(RETICULATE_PYTHON="/home/luca/anaconda3/envs/reticulate/bin/python")
library(reticulate)
reticulate::use_python("/home/luca/anaconda3/envs/reticulate/bin/python")
reticulate::use_condaenv("/home/luca/anaconda3/envs/reticulate")
reticulate::py_module_available(module='anndata') #needs to be TRUE
reticulate::import('anndata') #good to make sure this doesn't error

[1] TRUE

Module(anndata)


# Setup

## Load libraries

In [ ]:
pacman::p_load(CellChat, Seurat, patchwork, NMF, ggalluvial, dplyr, logr, parallel)

## Working Directories and Options

In [13]:
# Input files
Combine.Seurat = "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/Assets/20230518_RNA_FiltMin20exceptBetaLate_CellChat_nPODids.SCT.rds.SM.Lpct10.rds"
# Variables
condition.ls = c("ND", "Aab", "T1D_early","T1D_late")

# Output dirs
out.dir <- "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/Results/corrected/"


options(stringsAsFactors = FALSE)
# start log
options("logr.on" = TRUE, "logr.notes" = TRUE)
options("logr.autolog" = TRUE)
options("logr.compact" = TRUE)
options("logr.traceback" = TRUE)
log.file = paste(out.dir, Sys.Date(),".corrected.Run_CellChat.log", sep="")

In [14]:
log_open(log.file)

[1] "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/Results/corrected/log/2023-06-22.corrected.Run_CellChat.log"

# Pre Processing

In [15]:
log_print("Loading Seurat obj")
seurat_object <- readRDS(Combine.Seurat)

[1] "Loading Seurat obj"


In [16]:
# Check cell pops id
log_print("Changing idents")
seurat_object <- SetIdent(seurat_object, value = seurat_object$celltype)

[1] "Changing idents"


In [17]:
# Split object in conditions
seurat_object.ls = list()

log_print("Splitting conditions")
for (i in seq_along(condition.ls)){
    gc(reset = TRUE)
    condition.use = condition.ls[i]
    log_print(condition.use)
    seurat_object.ls[[i]] <- subset(x = seurat_object, subset = condition == condition.use)
}

[1] "Splitting conditions"
[1] "ND"
[1] "Aab"
[1] "T1D_early"
[1] "T1D_late"


In [18]:
sort(sapply(ls(),function(x){object.size(get(x))})) 
rm(seurat_object)
gc(reset = TRUE)

i    condition.use         log.file          out.dir 
              56              120              232              232 
  Combine.Seurat     condition.ls   multi.CellChat    seurat_object 
             240              320            64872        215528480 
seurat_object.ls 
       216063160

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,5300117,283.1,9659781,515.9,5300117,283.1
Vcells,27199282,207.6,68000469,518.9,27199282,207.6


## CellChat analysis

In [19]:
# Salva tutto in una directory

In [20]:
trim = 0
cores = 4 # I use 4 so that it will run 4 the conditions at the same time

In [24]:
multi.CellChat = function(i = 1){
    gc(reset = TRUE)
    # Setup variables
    seurat_object.use = seurat_object.ls[[i]]
    condition.use = condition.ls[[i]]
    log_print(paste("Starting CellChat analysis on: ", condition.use))
    # extract RNA data
    data.input <- GetAssayData(seurat_object.use, assay = "RNA", slot = "data") #get normalized data
    # Create CellChat Obj
    log_print(paste("Creating CellChat Obj: ", condition.use))
    labels <- Idents(seurat_object.use)
    meta <- data.frame(group = labels, row.names = names(labels)) # create a dataframe of the cell labels
    cellchat <- createCellChat(object = data.input, group.by = Idents(seurat_object.use))
    cellchat <- addMeta(cellchat, meta = meta, meta.name = "labels")
    cellchat <- setIdent(cellchat, ident.use = "labels")
    groupSize <- as.numeric(table(cellchat@idents)) # number of cells in each cell group
    CellChatDB <- CellChatDB.human # use CellChatDB.mouse if running on mouse data
    CellChatDB.use <- CellChatDB.human # use whole database
    cellchat@DB <- CellChatDB.use
    gc(reset = TRUE)
    # Running CellChat analysis
    log_print(paste("Identify interactions: ", condition.use))
    future::plan("multicore", workers = 5)
    cellchat <- subsetData(cellchat) # This step is necessary even if using the whole database
    cellchat <- identifyOverExpressedGenes(cellchat)
    cellchat <- identifyOverExpressedInteractions(cellchat)
    cellchat <- projectData(cellchat, PPI.human)
    gc(reset = TRUE)
    log_print(paste("Compute IS: ", condition.use))
    future::plan("multicore", workers = 5)
    options(future.globals.maxSize= 891289600)
    cellchat <- computeCommunProb(cellchat, type = "truncatedMean", trim = trim)
    gc(reset = TRUE)
    log_print(paste("Filter results: ", condition.use))
    # Filter out the cell-cell communication if there are only few number of cells in certain cell groups
    cellchat <- filterCommunication(cellchat, min.cells = 1)
    cellchat <- computeCommunProbPathway(cellchat)
    cellchat <- aggregateNet(cellchat)
    gc(reset = TRUE)
    log_print(paste("Save dfnet: ", condition.use))
    df.net <- subsetCommunication(cellchat,  thresh = 1)
    dfnet.file = paste(out.dir, condition.use, "_df.net_", trim, "_wholeccDB.txt", sep="")
    log_print(paste("Save RDS: ", condition.use))
    RDS.file = paste(out.dir, condition.use, "_", trim, "_wholeccDB.rds", sep="")
    saveRDS(cellchat, file = RDS.file)
    return(write.table(df.net, file=dfnet.file, sep = "\t", quote = F, col.names=T, row.names = F))
    log_print(paste("Saved: ", condition.use))
}

In [ ]:
mclapply(1:length(condition.ls), function(i) multi.CellChat(i), mc.cores = cores)